## Weighted grid sum

Compute $S = \sum_i \phi(x_i)\, (x_i \wedge (x_i \cdot B))$ over the
$10\times10\times10$ points of a 3-D grid, where $B$ is a bivector variable in
`BasisE3`, the $x_i$ are vectors in `BasisE3` and $\phi(x_i)$ is a scalar weight computed from the length of $x_i$
with an offset from the grid centre.

## Setup

In [8]:
import numpy as np
from pytanga import DataArray, Variable
from pytanga.basis import BasisE3
from pytanga.blade_mask import BladeMask

E3 = BasisE3()
point_mask = BladeMask(E3, grades=[1])
bivector_mask = BladeMask(E3, grades=[2])

B = Variable("B", bivector_mask)
x = Variable("x", point_mask)

grid = np.linspace(-1.0, 1.0, 10)
X, Y, Z = np.meshgrid(grid, grid, grid, indexing="ij")
points = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)

r = np.linalg.norm(points, axis=1)
phi = np.exp(-0.5 * ((r - 0.3) / 0.4) ** 2) / 1000

## Build and contract

In [9]:
expr = x ^ (x | B)

partial = expr(x=DataArray(points, masks=("pnt_idx", point_mask)))

S = partial(pnt_idx=phi)
S

Expression(names=['B'], out_mask=BladeMask(['e12', 'e13', 'e23']))

## The result is a 3x3 map

A bivector in `E3` has three coefficients, so the resulting expression tensor
is exactly a $3\times3$ matrix mapping the coefficients of `B` to the
coefficients of the result bivector.

In [10]:
M = S.tensor.data
M.shape, M

((3, 3),
 array([[ 1.07269420e-01, -1.46324942e-19,  1.39548678e-19],
        [-1.46324942e-19,  1.07269420e-01,  6.85461413e-19],
        [ 1.39548678e-19,  6.85461413e-19,  1.07269420e-01]]))

## Evaluating for a concrete B

In [ ]:
B_val = E3("e12 + 2 e13 + 3 e23")
# Now compute the result of the expression with this specific bivector value
result = S(B=B_val)
result

0.1073 e12 + 0.2145 e13 + 0.3218 e23